In [1]:
import os
import torch
import timm
import numpy as np
import pandas as pd
from torchvision import transforms
from PIL import Image
from sklearn.metrics import roc_auc_score, confusion_matrix, f1_score
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ==========================================
# 1. KAGGLE PATHS
# ==========================================
# Your dataset path
DATA_DIR = "/kaggle/input/datasets/prosenjitmondol/complete-vindr-spinexr/vindr-spinexr-a-large-annotated-medical-image-dataset/vindr-spinexr-a-large-annotated-medical-image-dataset"
TEST_CSV = os.path.join(DATA_DIR, 'annotations/test.csv')
TEST_DIR = os.path.join(DATA_DIR, 'test_png')

# Model weights
MODEL_DENSE_PATH = "/kaggle/input/datasets/prosenjitmondol/spine-ensemble/densenet.pth" 
MODEL_EFFICIENT_PATH = "/kaggle/input/datasets/prosenjitmondol/spine-ensemble/efficientnet.pth"
MODEL_RESNET_PATH = "/kaggle/input/datasets/prosenjitmondol/spine-ensemble/resnet50.pth"

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def load_model(model_name, checkpoint_path):
    print(f"Loading {model_name} from {checkpoint_path}...")
    if 'densenet' in model_name:
        model = timm.create_model('densenet121', pretrained=False, num_classes=1)
    elif 'efficientnet' in model_name:
        model = timm.create_model('tf_efficientnetv2_s', pretrained=False, num_classes=1)
    elif 'resnet' in model_name:
        model = timm.create_model('resnet50', pretrained=False, num_classes=1)
    
    # PYTORCH 2.6 FIX: Added weights_only=False to prevent UnpicklingError
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    
    if 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint)
    
    model.to(device).eval()
    return model

def predict_batch_tta(model, images, img_size=384):
    transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    batch = torch.stack([transform(img) for img in images]).to(device)
    with torch.no_grad():
        preds1 = torch.sigmoid(model(batch)).cpu().numpy().flatten()
    
    flipped = [transforms.functional.hflip(img) for img in images]
    batch_flip = torch.stack([transform(img) for img in flipped]).to(device)
    with torch.no_grad():
        preds2 = torch.sigmoid(model(batch_flip)).cpu().numpy().flatten()
    
    return (preds1 + preds2) / 2.0

# ==========================================
# 3. LOAD DATA & PREDICT
# ==========================================
print("="*80)
print("RUNNING EFFICIENTNETV2-S ABLATION STUDY")
print("="*80)

df = pd.read_csv(TEST_CSV)
image_labels = {}
# Assuming 'class_name' is used based on our previous fixes, or 'lesion_type' as in your script
label_col = 'class_name' if 'class_name' in df.columns else 'lesion_type'

for _, row in df.iterrows():
    img_id = row['image_id']
    lesion = row[label_col]
    if img_id not in image_labels:
        image_labels[img_id] = 0 if str(lesion).strip() == 'No finding' else 1
    elif str(lesion).strip() != 'No finding':
        image_labels[img_id] = 1

img_ids = list(image_labels.keys())
labels = np.array([image_labels[iid] for iid in img_ids])

# Load Models
model_dense = load_model('densenet', MODEL_DENSE_PATH)
model_eff = load_model('efficientnet', MODEL_EFFICIENT_PATH)
model_res = load_model('resnet', MODEL_RESNET_PATH)
models = [model_dense, model_eff, model_res]

# Get Predictions
print("\nGenerating TTA Predictions...")
batch_size = 28
all_preds = [[] for _ in range(3)]

for i in tqdm(range(0, len(img_ids), batch_size), desc="Processing Test Set"):
    batch_ids = img_ids[i:i+batch_size]
    # Handle paths safely
    images = []
    for iid in batch_ids:
        img_path = os.path.join(TEST_DIR, f"{iid}.png")
        if not os.path.exists(img_path):
            img_path = os.path.join(TEST_DIR, f"{iid}.dicom") # Fallback if needed
        images.append(Image.open(img_path).convert('RGB'))
    
    for mi, model in enumerate(models):
        preds = predict_batch_tta(model, images)
        all_preds[mi].extend(preds)

all_preds = np.array(all_preds)

# ==========================================
# 4. ABLATION ANALYSIS
# ==========================================
def evaluate_ensemble(preds, weights, threshold=0.478):
    ens_preds = np.average(preds, axis=0, weights=weights)
    binary = (ens_preds >= threshold).astype(int)
    
    auroc = roc_auc_score(labels, ens_preds) * 100
    tn, fp, fn, tp = confusion_matrix(labels, binary).ravel()
    sens = tp / (tp + fn) * 100
    spec = tn / (tn + fp) * 100
    return auroc, sens, spec, fp

# 1. Full 3-Model Ensemble (From your paper)
weights_full = [0.38, 0.36, 0.26] 
auroc_full, sens_full, spec_full, fp_full = evaluate_ensemble(all_preds, weights_full)

# 2. Ablated 2-Model Ensemble (No EfficientNet)
# We normalize the remaining weights: Dense (0.38) and Res (0.26)
dense_norm = 0.38 / (0.38 + 0.26) # ≈ 0.593
res_norm = 0.26 / (0.38 + 0.26)   # ≈ 0.407
weights_ablated = [dense_norm, res_norm]

# Only pass the predictions from DenseNet (idx 0) and ResNet (idx 2)
ablated_preds = np.array([all_preds[0], all_preds[2]])
auroc_ab, sens_ab, spec_ab, fp_ab = evaluate_ensemble(ablated_preds, weights_ablated)

print("\n" + "="*80)
print("ABLATION STUDY RESULTS (FOR REBUTTAL)")
print("="*80)
print(f"{'Configuration':<35} | {'AUROC':<8} | {'Sensitivity':<12} | {'Specificity':<12} | {'False Positives'}")
print("-" * 90)
print(f"{'Full DERNet (with EfficientNet)':<35} | {auroc_full:.2f}%  | {sens_full:.2f}%      | {spec_full:.2f}%      | {fp_full}")
print(f"{'Ablated DERNet (NO EfficientNet)':<35} | {auroc_ab:.2f}%  | {sens_ab:.2f}%      | {spec_ab:.2f}%      | {fp_ab}")
print("-" * 90)

fp_increase = fp_ab - fp_full
print(f"\nCONCLUSION:")
print(f"Removing EfficientNetV2-S caused Specificity to drop by {(spec_full - spec_ab):.2f}%.")
print(f"It also caused False Positives to increase by {fp_increase} cases.")

RUNNING EFFICIENTNETV2-S ABLATION STUDY
Loading densenet from /kaggle/input/datasets/prosenjitmondol/spine-ensemble/densenet.pth...
Loading efficientnet from /kaggle/input/datasets/prosenjitmondol/spine-ensemble/efficientnet.pth...
Loading resnet from /kaggle/input/datasets/prosenjitmondol/spine-ensemble/resnet50.pth...

Generating TTA Predictions...


Processing Test Set: 100%|██████████| 75/75 [10:24<00:00,  8.32s/it]


ABLATION STUDY RESULTS (FOR REBUTTAL)
Configuration                       | AUROC    | Sensitivity  | Specificity  | False Positives
------------------------------------------------------------------------------------------
Full DERNet (with EfficientNet)     | 90.97%  | 83.12%      | 82.99%      | 182
Ablated DERNet (NO EfficientNet)    | 90.82%  | 85.10%      | 81.21%      | 201
------------------------------------------------------------------------------------------

CONCLUSION:
Removing EfficientNetV2-S caused Specificity to drop by 1.78%.
It also caused False Positives to increase by 19 cases.
